# Carnet de démarrage — Mèche OS

Ce carnet charge les cours collectés par le projet Mèche, calcule des indicateurs et compare des actifs.
Exécutez les cellules une à une avec **Maj + Entrée**.

In [ ]:
import json, urllib.request
import pandas as pd
import matplotlib.pyplot as plt

BASE = 'https://raw.githubusercontent.com/bdembele192-pixel/Porte-de-la-russie/main/trading/data'

def charger(nom):
    with urllib.request.urlopen(f'{BASE}/{nom}.json', timeout=20) as r:
        return json.load(r)

cours = pd.DataFrame(charger('quotes')['items'])
historique = charger('history')['items']
cours[['id', 'name', 'type', 'price', 'chg', 'cur']].head(30)

## Historique journalier d'un actif

In [ ]:
def serie(symbole):
    barres = historique[symbole]
    df = pd.DataFrame(barres, columns=['t', 'ouverture', 'haut', 'bas', 'cloture', 'volume'])
    df.index = pd.to_datetime(df.pop('t'), unit='ms').dt.normalize()
    return df

aapl = serie('AAPL')
aapl['MM20'] = aapl['cloture'].rolling(20).mean()
aapl['MM50'] = aapl['cloture'].rolling(50).mean()
aapl[['cloture', 'MM20', 'MM50']].plot(figsize=(11, 4), title='Apple — clôture et moyennes mobiles')
plt.show()

## Rendements, volatilité et corrélations

In [ ]:
actifs = ['SPX', 'CAC', 'DAX', 'GOLD', 'BRENT', 'EURUSD']
clotures = pd.DataFrame({s: serie(s)['cloture'] for s in actifs}).dropna()
rendements = clotures.pct_change().dropna()

print('Volatilité annualisée (%) :')
print((rendements.std() * 252 ** 0.5 * 100).round(1))
rendements.corr().round(2)

In [ ]:
base100 = clotures / clotures.iloc[0] * 100
base100.plot(figsize=(11, 4), title='Performance comparée (base 100)')
plt.show()

## Pour aller plus loin

- `marche export SYMBOLE` dans un terminal crée un CSV lisible par `pd.read_csv(..., sep=';')`.
- statsmodels (régressions, ARIMA) et scikit-learn (classification, régression) sont déjà installés.

_Aide à l'analyse, pas un conseil en investissement._